# L10 Lab · Monitoring, Observability and Reporting

Companion notebook for [`monitoring.html`](./monitoring.html).

You will:
1. Run a tiny instrumented agent on a small golden set.
2. Read the traces it produces in a hand-rolled HTML viewer.
3. Deliberately break the prompt and watch which signal fires first.
4. Look at the anatomy of a useful weekly report.

No API keys required — the agent uses a deterministic mock LLM. See
`minimal_agent.py` for a commented block showing how to swap in a real
provider (OpenAI-compatible).

## 0. Install

Run this once. The lab only needs OpenTelemetry, Pydantic, and matplotlib.

In [ ]:
# %pip install opentelemetry-api opentelemetry-sdk pydantic matplotlib

## 1. Tour of the codebase

Open each of these in your editor as you go. The split is deliberate — the
agent is tiny, so the instrumentation stands out.

* `minimal_agent.py` — ~30 lines of actual agent logic.
* `06_observing_and_diagnosing.py` — JSON logging, OTel tracing, guardrails, runner,
  mock rubric judge, aggregate report.
* `trace_viewer.py` — stdlib-only HTML span-tree renderer (~150 lines).
* `weekly_report_demo.py` — charted 4-week fake dataset.
* `golden_set.json` — six test cases.

## 2. Run the agent on the golden set

This produces three artifacts:

* stdout — a stream of JSON log events (one per turn, one summary).
* `spans.jsonl` — one JSON span per line, for the trace viewer.
* `report.json` — aggregate metrics + per-record detail.

In [ ]:
!python 06_observing_and_diagnosing.py --prompt-version v1

## 3. Read the aggregate report

In [ ]:
import json
report = json.load(open('report.json'))
print(json.dumps(report['summary'], indent=2))

**Exercise 1 — read the signals.** Before going further:
* Which fields in `summary` are *leading* indicators (they will move before quality does)?
* Which are *lagging* (they move only once users are already hurt)?
* If you had to alert on exactly one of them, which would it be and why?

## 4. Render traces as HTML

The viewer is ~150 lines of stdlib. Read it after running — the code makes
the concept of a span tree very concrete.

In [ ]:
!python trace_viewer.py spans.jsonl --out trace.html
# Then open trace.html in a browser.
# If you're in Jupyter and want to inline the first trace:
from IPython.display import IFrame
IFrame('trace.html', width='100%', height=480)

## 5. Break the prompt — watch which signal catches it first

`--broken` injects malformed JSON roughly half the time. Compare the two
reports side-by-side: which aggregate metric moves the most, and which
guardrail starts doing the work?

In [ ]:
!python 06_observing_and_diagnosing.py --prompt-version v2 --broken --report report_broken.json --spans spans_broken.jsonl

In [ ]:
good = json.load(open('report.json'))['summary']
bad  = json.load(open('report_broken.json'))['summary']
print(f"{'metric':<28} {'good':>10} {'broken':>10}")
for k in ('mean_score', 'guardrail_reject_rate', 'repair_rate',
         'p50_latency_ms', 'p95_latency_ms', 'total_cost_usd'):
    print(f"{k:<28} {good[k]!s:>10} {bad[k]!s:>10}")

**Exercise 2 — which signal fired first?** Look at the row where `good` and
`broken` differ by the largest *relative* amount. That is almost certainly
the leading indicator you would want at the top of your dashboard. Write
down your answer before reading anyone else's.

## 6. Anatomy of a weekly report

Run the standalone demo. It emits a 3-panel chart and a three-sentence
prose summary — the minimal template the lecture argued for (*one quality
number with CI, one cost number, one safety number, short prose, one action
item*).

In [ ]:
!python weekly_report_demo.py --out weekly_report.png

In [ ]:
from IPython.display import Image
Image('weekly_report.png')

**Exercise 3 — write the prose yourself.** The script prints a sample
summary; ignore it and write your own first. Constraints: three sentences,
mention the CI, name the leading indicator, end with one action item.
Then compare to the script's output.

## 7. Deeper exercises

**Exercise 4 — third guardrail.** Add a "did the model actually call the
`search` tool when the question is out-of-vocabulary?" check. Wire it into
`run_instrumented` in `06_observing_and_diagnosing.py` as a third `guardrail.check`
span. Re-run and confirm the new verdict appears in both the log stream
and the HTML trace.

**Exercise 5 — move a rubric online.** Pick one rubric from L7 that you
think *should* run on every request. Justify it against the three
constraints from the lecture: cost per call, latency overhead, and failure
mode of the rubric itself.

**Exercise 6 — swap the exporter.** Uncomment the OTLP block in
`setup_tracing()`, install `opentelemetry-exporter-otlp-proto-http`, run a
local Phoenix or Jaeger (either works), and view the traces in that UI.
Compare what you see to your hand-rolled viewer. Where does each one win?

**Exercise 7 — design a CI eval gate.** Sketch, as pseudo-code or prose,
a `test_rubric_regression` pytest that (a) uses the uncertainty machinery
from L8–L9, (b) pins the dataset and judge versions, (c) fails the build
only on a statistically significant regression. You do not need to run it
— the design is the exercise.

---

*Designing Large Scale AI Systems · Spring 2026*